<a href="https://colab.research.google.com/github/ravipallisaisailesh/company-x-slow-stock/blob/main/Company_X_Slow_Stock_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files
from IPython.display import display

pd.set_option("display.max_columns", None)

print("Company X analysis started successfully!")

Company X analysis started successfully!


In [3]:
uploaded = files.upload()

file_name = list(uploaded.keys())[0]

print("Uploaded file:", file_name)

Saving Case_Data_Technology_and_Data_Management_All_Sheets.xlsx to Case_Data_Technology_and_Data_Management_All_Sheets.xlsx
Uploaded file: Case_Data_Technology_and_Data_Management_All_Sheets.xlsx


displays the name of your uploaded workbook

In [ ]:
file = pd.ExcelFile(file_name)

print("Sheets available:")
print(file.sheet_names)

items = pd.read_excel(file_name, sheet_name="Item_Master")
stock = pd.read_excel(file_name, sheet_name="Stock_Aging")
sales = pd.read_excel(file_name, sheet_name="Sales_Lines")
customers = pd.read_excel(file_name, sheet_name="Customers")

print("\nItem Master:", items.shape)
print("Stock Aging:", stock.shape)
print("Sales Lines:", sales.shape)
print("Customers:", customers.shape)

print("\nItem Master preview:")
display(items.head())

print("\nStock Aging preview:")
display(stock.head())

print("\nSales Lines preview:")
display(sales.head())

print("\nCustomers preview:")
display(customers.head())

The workbook you supplied contains 42 item records, 42 stock records, 630 sales records and 30 customer records.

In [ ]:
# Standardise SKU codes
for df in [items, stock, sales]:
    df["SKU"] = (
        df["SKU"]
        .astype("string")
        .str.strip()
        .str.upper()
    )

# Clean item master text
for col in ["Item Name", "Category", "Alternate Item (name)"]:
    items[col] = items[col].astype("string").str.strip()

items["Alternate Flag"] = (
    items["Alternate Flag"].astype("string").str.strip().str.upper()
)

items["Status"] = items["Status"].astype("string").str.strip().str.title()

# Clean customer identifiers and text
for df in [sales, customers]:
    df["CustomerID"] = df["CustomerID"].astype("string").str.strip()

for col in ["CustomerName", "Industry"]:
    sales[col] = sales[col].astype("string").str.strip()
    customers[col] = customers[col].astype("string").str.strip()

customers["Area"] = customers["Area"].astype("string").str.strip()

# Convert numeric columns
stock_cols = [
    "Qty 0-30 days",
    "Qty 31-60 days",
    "Qty 61-90 days",
    "Qty 91-180 days",
    "Qty 180+ days"
]

for col in stock_cols:
    stock[col] = pd.to_numeric(stock[col], errors="coerce")

items["Master Pack Qty"] = pd.to_numeric(
    items["Master Pack Qty"], errors="coerce"
)

items["Unit Cost (Rs)"] = pd.to_numeric(
    items["Unit Cost (Rs)"], errors="coerce"
)

sales["Qty"] = pd.to_numeric(sales["Qty"], errors="coerce")
sales["Value (Rs)"] = pd.to_numeric(
    sales["Value (Rs)"], errors="coerce"
)

print("Data cleaning completed.")

The case study expects you to detect problems, explain how your tool handles them and recommend fixes to the source data.

In [ ]:
problems = []

def report_problem(problem, details, handling, source_fix):
    problems.append({
        "Problem": problem,
        "Details": str(details),
        "Tool handling": handling,
        "Recommended source fix": source_fix
    })

# Duplicate item codes
duplicates = items[items["SKU"].duplicated(keep=False)]

if not duplicates.empty:
    report_problem(
        "Duplicate SKU in Item_Master",
        duplicates["SKU"].tolist(),
        "Flag for review because product details may be ambiguous.",
        "Maintain one authoritative record per SKU."
    )

# Negative stock quantities
negative_stock = stock[
    (stock[stock_cols] < 0).any(axis=1)
]

if not negative_stock.empty:
    report_problem(
        "Negative stock quantity",
        negative_stock[["SKU"] + stock_cols].to_dict("records"),
        "Flag negative values; do not silently change them to zero.",
        "Check stock adjustments, returns and the source export."
    )

# Missing customer industries
missing_industry = customers[
    customers["Industry"].isna()
    | customers["Industry"].eq("")
]

if not missing_industry.empty:
    report_problem(
        "Missing customer industry",
        missing_industry["CustomerID"].tolist(),
        "Do not use these customers for industry-based Bucket 2 matching.",
        "Complete the industry field in the customer master."
    )

# Inactive products that still have stock
stock_check = stock.merge(
    items[["SKU", "Status"]],
    on="SKU",
    how="left"
)

stock_check["Total Stock"] = stock_check[stock_cols].fillna(0).sum(axis=1)

inactive_stock = stock_check[
    (stock_check["Status"].str.lower() == "inactive")
    & (stock_check["Total Stock"] > 0)
]

if not inactive_stock.empty:
    report_problem(
        "Inactive items with stock",
        inactive_stock[["SKU", "Total Stock"]].to_dict("records"),
        "Exclude from active-item analysis and show a separate warning.",
        "Confirm whether to sell, transfer, write off or reactivate the stock."
    )

# Sales or stock SKU not found in the item master
master_skus = set(items["SKU"].dropna())

unknown_stock = sorted(set(stock["SKU"].dropna()) - master_skus)
unknown_sales = sorted(set(sales["SKU"].dropna()) - master_skus)

if unknown_stock:
    report_problem(
        "Stock SKU missing from Item_Master",
        unknown_stock,
        "Flag for correction; do not invent item costs or categories.",
        "Correct the SKU or add the missing master record."
    )

if unknown_sales:
    report_problem(
        "Sales SKU missing from Item_Master",
        unknown_sales,
        "Flag unmatched sales for investigation.",
        "Correct the SKU or add the missing master record."
    )

problems_df = pd.DataFrame(problems)

print("Number of detected problem types:", len(problems_df))
display(problems_df)

A data-quality check should report an error, not hide it. For example, a negative stock quantity needs investigation before anyone decides how to correct it

In [ ]:

print("Item Master columns:")
print(items.columns.tolist())

print("\nAlternate Flag values:")
print(items["Alternate Flag"].unique())

print("\nSample alternate items:")
display(items[["SKU", "Item Name", "Alternate Item (name)", "Alternate Flag"]].head(10))


In [ ]:

# Create a lookup from item name to SKU
name_to_sku = {}

for _, row in items.iterrows():
    if pd.notna(row["Item Name"]):
        key = str(row["Item Name"]).strip().casefold()
        name_to_sku[key] = row["SKU"]

# Create old SKU to new SKU mapping
switch_map = {}

for _, row in items.iterrows():

    flag = row["Alternate Flag"]

    # Skip missing flags safely
    if pd.isna(flag) or str(flag).strip().upper() != "SWITCH":
        continue

    alternate_name = row["Alternate Item (name)"]

    if pd.isna(alternate_name):
        print("Missing alternate name for:", row["SKU"])
        continue

    key = str(alternate_name).strip().casefold()
    new_sku = name_to_sku.get(key)

    if new_sku is None:
        print("Alternate item not found:", row["SKU"], "->", alternate_name)
    else:
        switch_map[row["SKU"]] = new_sku

print("Old SKU -> New SKU mappings:")

display(pd.DataFrame(
    list(switch_map.items()),
    columns=["Old SKU", "New SKU"]
))


Map old SKUs to new SKUs

In [ ]:
def final_sku(sku):

    current = sku
    visited = set()

    while current in switch_map:

        if current in visited:
            raise ValueError("Circular SKU switch detected: " + str(sku))

        visited.add(current)
        current = switch_map[current]

    return current


items["Final SKU"] = items["SKU"].apply(final_sku)
stock["Final SKU"] = stock["SKU"].apply(final_sku)
sales["Final SKU"] = sales["SKU"].apply(final_sku)

print("Pack-size mapping completed.")

It records which SKU replaces which. The next step uses that mapping to combine the quantities correctly.

In [ ]:
# Fill missing bucket values with zero for calculations,
# while the earlier checks retain the issue in the report.
stock[stock_cols] = stock[stock_cols].fillna(0)

stock_combined = (
    stock.groupby("Final SKU", as_index=False)[stock_cols]
    .sum()
)

print("Combined stock preview:")
display(stock_combined.head())

Combine stock under the new SKU

In [ ]:
sales["Qty"] = sales["Qty"].fillna(0)
sales["Value (Rs)"] = sales["Value (Rs)"].fillna(0)

sales_combined = (
    sales.groupby("Final SKU", as_index=False)
    .agg(
        Total_Sold_Qty=("Qty", "sum"),
        Total_Sales_Value_Rs=("Value (Rs)", "sum")
    )
)

print("Combined sales preview:")
display(sales_combined.head())

Combine six-month sales under the new SKU

In [ ]:
# Keep only final SKUs for active products.
# Old switch SKUs are represented by their replacement SKU.
active_items = items[
    (items["Status"].str.lower() == "active")
    & (items["SKU"] == items["Final SKU"])
].copy()

analysis = (
    active_items
    .merge(stock_combined, on="Final SKU", how="left")
    .merge(sales_combined, on="Final SKU", how="left")
)

analysis[stock_cols] = analysis[stock_cols].fillna(0)

analysis["Total_Sold_Qty"] = (
    analysis["Total_Sold_Qty"].fillna(0)
)

analysis["Total_Sales_Value_Rs"] = (
    analysis["Total_Sales_Value_Rs"].fillna(0)
)

# Calculate the key stock metrics
analysis["Total Stock"] = analysis[stock_cols].sum(axis=1)

analysis["Aged Qty"] = (
    analysis["Qty 91-180 days"]
    + analysis["Qty 180+ days"]
)

analysis["Aged Value (Rs)"] = (
    analysis["Aged Qty"] * analysis["Unit Cost (Rs)"]
)

analysis["Average Monthly Sales"] = (
    analysis["Total_Sold_Qty"] / 6
)

analysis["Months to Clear"] = np.where(
    analysis["Average Monthly Sales"] > 0,
    analysis["Total Stock"] / analysis["Average Monthly Sales"],
    np.inf
)

print("Stock and sales calculations completed.")

display(
    analysis[[
        "Final SKU",
        "Item Name",
        "Total Stock",
        "Aged Qty",
        "Aged Value (Rs)",
        "Total_Sold_Qty",
        "Average Monthly Sales",
        "Months to Clear"
    ]].head(10)
)

Create the item analysis table

In [ ]:
def classify_product(row):

    # Rule 1: Red List
    if row["Total_Sold_Qty"] < row["Master Pack Qty"]:
        return "Red List"

    # Rule 2: No Action
    if (
        row["Aged Qty"] == 0
        or row["Months to Clear"] < 3
    ):
        return "No Action"

    # Rule 3: Push
    return "Push"


analysis["Class"] = analysis.apply(
    classify_product,
    axis=1
)

print("Product classification completed.")

display(
    analysis[[
        "Final SKU",
        "Item Name",
        "Total_Sold_Qty",
        "Master Pack Qty",
        "Aged Qty",
        "Months to Clear",
        "Class"
    ]]
)

Apply the classification rules

In [ ]:
class_summary = (
    analysis["Class"]
    .value_counts()
    .reindex(["Red List", "No Action", "Push"], fill_value=0)
    .rename_axis("Class")
    .reset_index(name="Number of Items")
)

push_items = analysis[
    analysis["Class"] == "Push"
].copy()

total_push_aged_value = push_items["Aged Value (Rs)"].sum()

top_10_push = (
    push_items
    .sort_values("Aged Value (Rs)", ascending=False)
    .head(10)
)

print("Number of items in each class:")
display(class_summary)

print(
    f"Total aged value of Push items: "
    f"₹{total_push_aged_value:,.2f}"
)

print("Top 10 Push items by aged value:")
display(
    top_10_push[[
        "Final SKU",
        "Item Name",
        "Category",
        "Total Stock",
        "Aged Qty",
        "Aged Value (Rs)",
        "Months to Clear"
    ]]
)

Calculate the management summary

In [ ]:
customer_spend = (
    sales.groupby("CustomerID", as_index=False)
    .agg(
        Customer_Spend_6M=("Value (Rs)", "sum")
    )
)

customer_info = customers.merge(
    customer_spend,
    on="CustomerID",
    how="left"
)

customer_info["Customer_Spend_6M"] = (
    customer_info["Customer_Spend_6M"].fillna(0)
)

eligible_customers = customer_info[
    customer_info["Customer_Spend_6M"] > 100000
].copy()

print("Eligible customers spending more than ₹1,00,000:")
print(len(eligible_customers))

display(
    eligible_customers.sort_values(
        "Customer_Spend_6M",
        ascending=False
    ).head(10)
)

Calculate customer spending

In [ ]:
# Look up the category of each final SKU
category_lookup = (
    analysis.drop_duplicates("Final SKU")
    .set_index("Final SKU")["Category"]
    .to_dict()
)

# Map each customer sale to its final SKU category
sales["Final Category"] = sales["Final SKU"].map(category_lookup)

# Customer IDs who bought each final SKU
buyers_by_sku = (
    sales.groupby("Final SKU")["CustomerID"]
    .apply(lambda values: set(values.dropna()))
    .to_dict()
)

# Industries of customers who bought each final SKU
industry_by_sku = {}

for sku, group in sales.groupby("Final SKU"):

    buyer_ids = set(group["CustomerID"].dropna())

    # Prefer customer-master industry information.
    buyer_info = customers[
        customers["CustomerID"].isin(buyer_ids)
    ]

    industries = set(
        buyer_info["Industry"]
        .dropna()
        .astype(str)
        .str.strip()
    )

    # Use sales-line industry when no master industry is recorded.
    sales_industries = set(
        group["Industry"]
        .dropna()
        .astype(str)
        .str.strip()
    )

    industry_by_sku[sku] = {
        value for value in industries.union(sales_industries)
        if value and value.casefold() != "nan"
    }

# Categories bought by each customer
categories_by_customer = (
    sales.groupby("CustomerID")["Final Category"]
    .apply(lambda values: set(values.dropna()))
    .to_dict()
)

eligible_ids = set(eligible_customers["CustomerID"].dropna())

# Customer information lookup
customer_lookup = (
    customer_info.drop_duplicates("CustomerID")
    .set_index("CustomerID")
    .to_dict("index")
)

target_rows = []

for _, item in push_items.iterrows():

    sku = item["Final SKU"]
    category = item["Category"]

    buyers = buyers_by_sku.get(sku, set())
    industries = industry_by_sku.get(sku, set())

    for customer_id in eligible_ids:

        customer = customer_lookup[customer_id]
        industry = customer.get("Industry")

        if pd.isna(industry) or str(industry).strip() == "":
            industry = None
        else:
            industry = str(industry).strip()

        bought_same_category = (
            category in categories_by_customer.get(customer_id, set())
        )

        # Apply buckets in priority order.
        if customer_id in buyers:
            bucket = 1
            bucket_name = "Bought before"

        elif industry is not None and industry in industries:
            bucket = 2
            bucket_name = "Similar business"

        elif bought_same_category:
            bucket = 3
            bucket_name = "Same category"

        else:
            continue

        target_rows.append({
            "SKU": sku,
            "Item Name": item["Item Name"],
            "Category": category,
            "CustomerID": customer_id,
            "CustomerName": customer.get("CustomerName"),
            "Industry": industry,
            "Area": customer.get("Area"),
            "Customer Spend 6M (Rs)": customer["Customer_Spend_6M"],
            "Bucket": bucket,
            "Bucket Name": bucket_name
        })

customer_targets = pd.DataFrame(target_rows)

if not customer_targets.empty:

    customer_targets = customer_targets.sort_values(
        ["SKU", "Bucket", "Customer Spend 6M (Rs)"],
        ascending=[True, True, False]
    ).reset_index(drop=True)

print("Customer targeting completed.")

display(customer_targets.head(20))

Build the customer targeting list

In [ ]:
required_skus = ["SKU1001", "SKU1006", "SKU1019"]

sku_answers = analysis[
    analysis["Final SKU"].isin(required_skus)
].copy()

sku_answers["Reason"] = sku_answers.apply(
    lambda row: (
        f"Sold less than one master pack "
        f"({row['Total_Sold_Qty']:.0f} sold vs "
        f"{row['Master Pack Qty']:.0f} per pack)."
        if row["Class"] == "Red List"
        else (
            "No aged stock."
            if row["Aged Qty"] == 0
            else (
                f"Months to clear = {row['Months to Clear']:.2f}, "
                "below 3."
                if row["Months to Clear"] < 3
                else (
                    f"Aged quantity = {row['Aged Qty']:.0f}; "
                    f"months to clear = {row['Months to Clear']:.2f}."
                )
            )
        )
    ),
    axis=1
)

display(
    sku_answers[[
        "Final SKU",
        "Class",
        "Total_Sold_Qty",
        "Master Pack Qty",
        "Aged Qty",
        "Months to Clear",
        "Reason"
    ]]
)

Find the three SKU classifications

In [ ]:
sku1005_targets = (
    customer_targets[
        customer_targets["SKU"] == "SKU1005"
    ]
    .sort_values(
        ["Bucket", "Customer Spend 6M (Rs)"],
        ascending=[True, False]
    )
    .head(3)
)

print("First three customers to call for SKU1005:")

display(
    sku1005_targets[[
        "CustomerID",
        "CustomerName",
        "Industry",
        "Customer Spend 6M (Rs)",
        "Bucket",
        "Bucket Name"
    ]]
)

Find the first three customers for SKU1005

In [ ]:
# Summary metrics
summary = pd.DataFrame([
    {
        "Metric": "Active final SKUs analysed",
        "Value": len(analysis)
    },
    {
        "Metric": "Red List items",
        "Value": (analysis["Class"] == "Red List").sum()
    },
    {
        "Metric": "No Action items",
        "Value": (analysis["Class"] == "No Action").sum()
    },
    {
        "Metric": "Push items",
        "Value": (analysis["Class"] == "Push").sum()
    },
    {
        "Metric": "Total aged value of Push items (Rs)",
        "Value": total_push_aged_value
    },
    {
        "Metric": "Eligible customers",
        "Value": len(eligible_customers)
    }
])

# Top 10 Push products
top10 = push_items.sort_values(
    "Aged Value (Rs)", ascending=False
).head(10)

# Save all outputs into one Excel workbook
output_file = "Company_X_Slow_Stock_Results.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    summary.to_excel(
        writer, sheet_name="Summary", index=False
    )

    class_summary.to_excel(
        writer, sheet_name="Class_Counts", index=False
    )

    analysis.to_excel(
        writer, sheet_name="Item_List", index=False
    )

    top10.to_excel(
        writer, sheet_name="Top_10_Push", index=False
    )

    customer_targets.to_excel(
        writer, sheet_name="Customer_Targets", index=False
    )

    pd.DataFrame(problems).to_excel(
        writer, sheet_name="Data_Problems", index=False
    )

    inactive_stock.to_excel(
        writer, sheet_name="Inactive_Stock", index=False
    )

    sku_answers.to_excel(
        writer, sheet_name="Specified_SKUs", index=False
    )

    sku1005_targets.to_excel(
        writer, sheet_name="SKU1005_Top_3", index=False
    )

    pd.DataFrame(
        list(switch_map.items()),
        columns=["Old SKU", "New SKU"]
    ).to_excel(
        writer, sheet_name="Switch_Mapping", index=False
    )

print("Excel results created successfully!")

files.download(output_file)